In [ ]:
import cv2
import matplotlib.pyplot as plt
from roboflow import Roboflow

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

!yolo task=detect mode=train model="runs/detect/train11/weights/last.pt" data = "C:\Users\Rylle\Downloads\YOLOv8_Trash_Object_Detection\YOLOv8_Trash_Object_Detection\data.yaml" epochs=600 imgsz=640 batch=8 device=0 workers=0

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

!yolo task=detect mode=train model="runs/detect/train12/weights/last.pt" data = "C:\Users\Rylle\Downloads\YOLOv8_Trash_Object_Detection\yolov8-trash-detections.v6i.yolov8\data.yaml" epochs=250 imgsz=640 batch=8 device=0 workers=0

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

!yolo task=detect mode=train \
    model="C:\Users\Rylle\Downloads\YOLOv8_Trash_Object_Detection\YOLOv8_Trash_Object_Detection\runs\detect\train19\weights\last.pt" \
    data="C:/Users/Rylle/Downloads/YOLOv8_Trash_Object_Detection/yolov8-trash-detections.v6i.yolov8/data.yaml" \
    epochs=150 \
    imgsz=640 \
    batch=16 \
    device=0 \
    workers=4 \
    save_period=10 \
    patience=20 \
    optimizer=AdamW \
    close_mosaic=10 \
    half=True


In [ ]:
import cv2
import yaml
from ultralytics import YOLO

# Load the trained YOLOv8 model
model = YOLO("runs/detect/train13/weights/best.pt")  # Use your trained model path

# Load class names from data.yaml dynamically
with open(r"C:\Users\Rylle\Downloads\YOLOv8_Trash_Object_Detection\YOLOv8_Trash_Object_Detection\data.yaml", "r") as f:
    data = yaml.safe_load(f)
class_names = data["names"]  # Extract class names dynamically

cap = cv2.VideoCapture(1)  # Start webcam feed

while True:
    ret, frame = cap.read()
    
    if not ret:
        print("Failed to capture frame.")
        break
        
    # Perform inference using the model
    result = model(frame)  # Run the frame through the trained model

    # The result will contain predictions
    predictions = result[0].boxes  # Get bounding box predictions
    
    for box in predictions:
        # Get box coordinates, class, and confidence score
        x1, y1, x2, y2 = box.xyxy[0]  # Bounding box coordinates
        confidence = box.conf[0]  # Confidence score
        class_id = int(box.cls[0])  # Class ID (numerical index)

        # Draw bounding box
        cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
        
        # Get class name dynamically from the dataset
        class_name = class_names[class_id] if class_id < len(class_names) else f"Class_{class_id}"

        # Put class label and confidence
        cv2.putText(frame, f"{class_name} ({confidence:.2f})", (int(x1), int(y1) - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    # Display the frame with predictions
    cv2.imshow('Webcam Feed', frame)

    # Wait for keypress to exit
    if cv2.waitKey(1) == ord('q'):
        break
    if cv2.waitKey(1) == ord(' '):
        cv2.waitKey(-1)

cap.release()
cv2.destroyAllWindows()

In [ ]:
import cv2

# Try indexes 0 to 10 to find Camo's index
for i in range(10):
    cap = cv2.VideoCapture(i)
    if cap.isOpened():
        print(f"Camera index {i} is available.")
        cap.release()


In [ ]:
import cv2
import tkinter as tk
from tkinter import filedialog, ttk
from PIL import Image, ImageTk
from ultralytics import YOLO

# Load the trained YOLOv8 model
model = YOLO("runs/detect/train13/weights/best.pt")

# Initialize main window
root = tk.Tk()
root.title("Trash Detection System")
root.geometry("900x800")  # Adjusted size
root.configure(bg="#2C3E50")  # Dark theme

# Default display area size
display_width, display_height = 720, 480

# Function to create a blank black image
def create_black_image(width, height):
    black_img = Image.new("RGB", (width, height), "black")
    return ImageTk.PhotoImage(black_img)

# Default black screen
black_screen = create_black_image(display_width, display_height)

# Header Frame
header_frame = tk.Frame(root, bg="#1ABC9C", height=50)
header_frame.pack(fill="x")

title_label = tk.Label(header_frame, text="Trash Detection System",
                       font=("Arial", 20, "bold"), fg="white", bg="#1ABC9C", pady=10)
title_label.pack()

# Video Frame (For image/webcam feed)
video_frame = tk.Frame(root, bg="#34495E", bd=5)
video_frame.pack(pady=20)

label_img = tk.Label(video_frame, image=black_screen, width=display_width, height=display_height)
label_img.pack()

# VideoCapture object (global)
cap = None
selected_camera = tk.IntVar(value=0)  # Default: Laptop Webcam (0)

# Function to resize image to fit display area while maintaining aspect ratio
def resize_image(image, max_width, max_height):
    img_width, img_height = image.size

    # Calculate scaling factor
    scale_factor = min(max_width / img_width, max_height / img_height)

    # Resize image
    new_width = int(img_width * scale_factor)
    new_height = int(img_height * scale_factor)

    return image.resize((new_width, new_height), Image.Resampling.LANCZOS)

# Image Upload Function
def upload_img():
    file_path = filedialog.askopenfilename(filetypes=[("Image Files", ".jpg;.png;*.jpeg;*")])
    if file_path:
        detect_image(file_path)

# Image Detection Function
def detect_image(path):
    results = model(path)
    result_img = results[0].plot()

    img = cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB)
    img = Image.fromarray(img)
    img = resize_image(img, display_width, display_height)  # Resize to fit
    img = ImageTk.PhotoImage(img)

    label_img.config(image=img)
    label_img.image = img

# Function to update webcam feed inside GUI
def update_webcam():
    global cap
    if cap is None or not cap.isOpened():
        return

    ret, frame = cap.read()
    if not ret:
        return

    # Inference
    result = model(frame)
    predictions = result[0].boxes

    for box in predictions:
        x1, y1, x2, y2 = box.xyxy[0]
        confidence = box.conf[0]  
        class_id = int(box.cls[0])

        cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
        class_name = f"Class_{class_id}"
        cv2.putText(frame, f"{class_name} ({confidence:.2f})", (int(x1), int(y1) - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    # Convert OpenCV frame to Tkinter-compatible format
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    frame = Image.fromarray(frame)
    frame = resize_image(frame, display_width, display_height)  # Resize to fit
    frame = ImageTk.PhotoImage(frame)

    label_img.config(image=frame)
    label_img.image = frame

    # Repeat function after 10ms
    root.after(10, update_webcam)

# Function to start webcam inside GUI
def start_webcam():
    global cap

    # Release existing camera if any
    if cap:
        cap.release()
        cap = None

    # Get selected camera index
    cam_index = selected_camera.get()

    cap = cv2.VideoCapture(cam_index)

    update_webcam()

# Function to stop webcam and reset to black screen
def stop_webcam():
    global cap
    if cap:
        cap.release()
        cap = None
        label_img.config(image=black_screen)
        label_img.image = black_screen

# Button Styling
button_style = {
    "font": ("Arial", 12, "bold"),
    "fg": "white",
    "bg": "#2980B9",
    "activebackground": "#1F618D",
    "activeforeground": "white",
    "bd": 3,
    "relief": "raised",
    "width": 15,
    "height": 2
}

# Camera Selection Frame
camera_frame = tk.Frame(root, bg="#2C3E50")
camera_frame.pack(pady=10)

camera_label = tk.Label(camera_frame, text="Select Camera:",
                        font=("Arial", 12, "bold"), fg="white", bg="#2C3E50")
camera_label.grid(row=0, column=0, padx=10, pady=5)

camera_options = [("Laptop Webcam (0)", 0), ("External Camera (1)", 1)]
for text, value in camera_options:
    rb = tk.Radiobutton(camera_frame, text=text, variable=selected_camera, value=value,
                        font=("Arial", 12), fg="white", bg="#2C3E50", selectcolor="#1ABC9C")
    rb.grid(row=0, column=value+1, padx=10, pady=5)

# Button Frame
btn_frame = tk.Frame(root, bg="#2C3E50")
btn_frame.pack(pady=10)

# Buttons
btn_upload = tk.Button(btn_frame, text="Upload Image", command=upload_img, **button_style)
btn_upload.grid(row=0, column=0, padx=10, pady=5)

btn_webcam = tk.Button(btn_frame, text="Start Webcam", command=start_webcam, **button_style)
btn_webcam.grid(row=0, column=1, padx=10, pady=5)

btn_stop_webcam = tk.Button(btn_frame, text="Stop Webcam", command=stop_webcam, **button_style)
btn_stop_webcam.grid(row=0, column=2, padx=10, pady=5)

root.mainloop()


In [ ]:
import cv2
import tkinter as tk
from tkinter import filedialog, ttk
from PIL import Image, ImageTk
from ultralytics import YOLO

# YOLOv8 model
model = YOLO("runs/detect/train19/weights/best.pt")

# Main window
root = tk.Tk()
root.title("Trash Detection System")
root.geometry("900x800")  # Adjusted size
root.configure(bg="#2C3E50")  # Dark theme

# Default display area size
display_width = 720
display_height = 480

# Function to create a blank black image
def create_black_image(width, height):
    black_img = Image.new("RGB", (width, height), "black")
    return ImageTk.PhotoImage(black_img)

# Default black screen
black_screen = create_black_image(display_width, display_height)

# Header Frame
header_frame = tk.Frame(root, bg="#1ABC9C", height=50)
header_frame.pack(fill="x")

title_label = tk.Label(header_frame,
                       text="Trash Detection System",
                       font=("Arial", 20, "bold"),
                       fg="white",
                       bg="#1ABC9C",
                       pady=10
                      )
title_label.pack()

# Video Frame (For image/webcam feed)
video_frame = tk.Frame(root, bg="#34495E", bd=5)
video_frame.pack(pady=20)

label_img = tk.Label(video_frame,
                     image=black_screen,
                     width=display_width,
                     height=display_height
                    )
label_img.pack()

# VideoCapture object (global)
cap = None
selected_camera = tk.IntVar(value=0)  # Default: Laptop Webcam (0)

# Function to resize image to fit display area while maintaining aspect ratio
def resize_image(image, max_width, max_height):
    img_width, img_height = image.size

    # Calculate scaling factor
    scale_factor = min(max_width / img_width, max_height / img_height)

    # Resize image
    new_width = int(img_width * scale_factor)
    new_height = int(img_height * scale_factor)

    return image.resize((new_width, new_height), Image.Resampling.LANCZOS)

# Image Upload
def upload_img():
    file_path = filedialog.askopenfilename(filetypes=[("Image Files", ".jpg;.png;*.jpeg;*")])
    if file_path:
        detect_image(file_path)

# Image Detection
def detect_image(path):
    results = model(path)
    result_img = results[0].plot()

    img = cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB)
    img = Image.fromarray(img)
    img = resize_image(img, display_width, display_height)  # Resize to fit
    img = ImageTk.PhotoImage(img)

    label_img.config(image=img)
    label_img.image = img

# Function to update webcam feed inside GUI
def update_webcam():
    global cap
    if cap is None or not cap.isOpened():
        return

    ret, frame = cap.read()
    if not ret:
        return

    # Inference
    result = model(frame)
    predictions = result[0].boxes

    for box in predictions:
        x1, y1, x2, y2 = box.xyxy[0]
        confidence = box.conf[0]  
        class_id = int(box.cls[0])

        cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
        class_name = model.names[class_id]
        cv2.putText(frame, f"{class_name} ({confidence:.2f})", (int(x1), int(y1) - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    # Convert OpenCV frame to Tkinter-compatible format
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    frame = Image.fromarray(frame)
    frame = resize_image(frame, display_width, display_height)  # Resize to fit
    frame = ImageTk.PhotoImage(frame)

    label_img.config(image=frame)
    label_img.image = frame

    # Repeat function after 10ms
    root.after(10, update_webcam)

# Function to start webcam inside GUI
def start_webcam():
    global cap

    # Release existing camera if any
    if cap:
        cap.release()
        cap = None

    # Get selected camera index
    cam_index = selected_camera.get()

    cap = cv2.VideoCapture(cam_index)

    update_webcam()

# Function to stop webcam and reset to black screen
def stop_webcam():
    global cap
    if cap:
        cap.release()
        cap = None
        label_img.config(image=black_screen)
        label_img.image = black_screen

# Button Styling
button_style = {
    "font": ("Arial", 12, "bold"),
    "fg": "white",
    "bg": "#2980B9",
    "activebackground": "#1F618D",
    "activeforeground": "white",
    "bd": 3,
    "relief": "raised",
    "width": 15,
    "height": 2
}

# Camera Selection Frame
camera_frame = tk.Frame(root,
                        bg="#2C3E50"
                       )
camera_frame.pack(pady=10)

camera_label = tk.Label(camera_frame,
                        text="Select Camera:",
                        font=("Arial", 12, "bold"),
                        fg="white",
                        bg="#2C3E50"
                       )
camera_label.grid(row=0, column=0, padx=10, pady=5)

camera_options = [("Laptop Webcam (0)", 0), ("External Camera (1)", 1)]
for text, value in camera_options:
    rb = tk.Radiobutton(camera_frame,
                        text=text,
                        variable=selected_camera,
                        value=value,
                        font=("Arial", 12),
                        fg="white",
                        bg="#2C3E50",
                        selectcolor="#1ABC9C"
                       )
    rb.grid(row=0,
            column=value+1,
            padx=10,
            pady=5
           )

# Button Frame
btn_frame = tk.Frame(root,
                     bg="#2C3E50"
                    )
btn_frame.pack(pady=10)

# Buttons
btn_upload = tk.Button(btn_frame,
                       text="Upload Image",
                       command=upload_img,
                       **button_style
                      )
btn_upload.grid(row=0,
                column=0,
                padx=10,
                pady=5
               )

btn_webcam = tk.Button(btn_frame,
                       text="Start Webcam",
                       command=start_webcam,
                       **button_style
                      )
btn_webcam.grid(row=0,
                column=1,
                padx=10,
                pady=5
               )

btn_stop_webcam = tk.Button(btn_frame,
                            text="Stop Webcam",
                            command=stop_webcam,
                            **button_style
                           )
btn_stop_webcam.grid(row=0,
                     column=2,
                     padx=10,
                     pady=5
                    )

root.mainloop()